# Time Series Fundamentals

Core concepts for time series analysis: decomposition, stationarity, ACF/PACF, and exponential smoothing.

## Overview

This notebook covers:
- Creating synthetic time series with trend and seasonality
- Time series decomposition
- Autocorrelation (ACF) and partial autocorrelation (PACF)
- Stationarity testing
- Differencing
- Exponential smoothing methods
- Train/test splitting for time series

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy import signal

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')


## 1. Create Synthetic Time Series Data

In [ ]:
# Create time series with trend, seasonality, and noise
n_steps = 200
t = np.arange(n_steps)

# Components
trend = 0.5 * t
seasonality = 10 * np.sin(2 * np.pi * t / 12) + 5 * np.cos(4 * np.pi * t / 12)
noise = np.random.normal(0, 2, n_steps)

# Combine components
y = trend + seasonality + noise

print(f'Time series length: {len(y)}')
print(f'Mean: {np.mean(y):.2f}')
print(f'Std Dev: {np.std(y):.2f}')
print(f'Min: {np.min(y):.2f}, Max: {np.max(y):.2f}')


## 2. Plot Time Series

In [ ]:
# Plot the time series
fig, axes = plt.subplots(4, 1, figsize=(12, 10))

axes[0].plot(t, y, linewidth=1.5, alpha=0.8)
axes[0].set_title('Full Time Series (Trend + Seasonality + Noise)')
axes[0].set_ylabel('Value')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, trend, label='Trend', linewidth=2)
axes[1].set_title('Trend Component')
axes[1].set_ylabel('Value')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(t, seasonality, label='Seasonality', color='orange', linewidth=2)
axes[2].set_title('Seasonal Component')
axes[2].set_ylabel('Value')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

axes[3].plot(t, noise, label='Noise', color='red', linewidth=1, alpha=0.7)
axes[3].set_title('Noise Component')
axes[3].set_ylabel('Value')
axes[3].set_xlabel('Time Step')
axes[3].legend()
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 3. Time Series Decomposition (Manual)

In [ ]:
# Manual decomposition using moving average
window_size = 12  # 12-month seasonality

# Trend: moving average
trend_ma = np.convolve(y, np.ones(window_size)/window_size, mode='same')

# Detrended: remove trend
detrended = y - trend_ma

# Seasonal: average of detrended by season
seasonal = np.zeros_like(y)
for season in range(window_size):
    season_indices = np.arange(season, n_steps, window_size)
    if len(season_indices) > 0:
        season_avg = np.mean(detrended[season_indices])
        seasonal[season_indices] = season_avg

# Residual
residual = y - trend_ma - seasonal

# Plot decomposition
fig, axes = plt.subplots(4, 1, figsize=(12, 10))

axes[0].plot(t, y, label='Original', linewidth=1.5, alpha=0.8)
axes[0].set_title('Original Time Series')
axes[0].set_ylabel('Value')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, trend_ma, label='Trend (MA)', linewidth=2, color='orange')
axes[1].set_title('Trend Component')
axes[1].set_ylabel('Value')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(t, seasonal, label='Seasonal', linewidth=2, color='green')
axes[2].set_title('Seasonal Component')
axes[2].set_ylabel('Value')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

axes[3].plot(t, residual, label='Residual', linewidth=1, color='red', alpha=0.7)
axes[3].set_title('Residual Component')
axes[3].set_ylabel('Value')
axes[3].set_xlabel('Time Step')
axes[3].legend()
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 4. Autocorrelation Function (ACF)

In [ ]:
# Compute ACF manually using numpy
def compute_acf(series, nlags=40):
    """Compute autocorrelation function"""
    series = np.asarray(series)
    series = series - np.mean(series)
    
    c0 = np.dot(series, series) / len(series)
    acf_values = np.zeros(nlags + 1)
    acf_values[0] = 1.0
    
    for k in range(1, nlags + 1):
        c_k = np.dot(series[:-k], series[k:]) / len(series)
        acf_values[k] = c_k / c0
    
    return acf_values

acf_values = compute_acf(y, nlags=40)
lags = np.arange(len(acf_values))

# Confidence interval (95%)
conf_int = 1.96 / np.sqrt(len(y))

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(lags, acf_values, width=0.5, label='ACF')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.axhline(y=conf_int, color='red', linestyle='--', linewidth=1, label='95% Confidence Interval')
ax.axhline(y=-conf_int, color='red', linestyle='--', linewidth=1)
ax.set_xlabel('Lag')
ax.set_ylabel('ACF')
ax.set_title('Autocorrelation Function (ACF)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 5. Partial Autocorrelation Function (PACF)

In [ ]:
# Compute PACF using Yule-Walker equations
def compute_pacf(series, nlags=40):
    """Compute partial autocorrelation function using Yule-Walker"""
    acf_vals = compute_acf(series, nlags)
    pacf_values = np.zeros(nlags + 1)
    pacf_values[0] = 1.0
    
    for k in range(1, nlags + 1):
        # Simplified PACF computation
        if k == 1:
            pacf_values[k] = acf_vals[1]
        else:
            numerator = acf_vals[k]
            for j in range(1, k):
                numerator -= pacf_values[j] * acf_vals[k - j]
            
            denominator = 1.0
            for j in range(1, k):
                denominator -= pacf_values[j] * acf_vals[j]
            
            if denominator != 0:
                pacf_values[k] = numerator / denominator
    
    return pacf_values

pacf_values = compute_pacf(y, nlags=40)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(lags, acf_values, width=0.5)
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0].axhline(y=conf_int, color='red', linestyle='--', linewidth=1)
axes[0].axhline(y=-conf_int, color='red', linestyle='--', linewidth=1)
axes[0].set_xlabel('Lag')
axes[0].set_ylabel('ACF')
axes[0].set_title('Autocorrelation Function (ACF)')
axes[0].grid(True, alpha=0.3)

axes[1].bar(lags, pacf_values, width=0.5, color='orange')
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1].axhline(y=conf_int, color='red', linestyle='--', linewidth=1)
axes[1].axhline(y=-conf_int, color='red', linestyle='--', linewidth=1)
axes[1].set_xlabel('Lag')
axes[1].set_ylabel('PACF')
axes[1].set_title('Partial Autocorrelation Function (PACF)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 6. Stationarity Check

In [ ]:
# Check stationarity by examining rolling mean and std
window = 12
rolling_mean = pd.Series(y).rolling(window=window).mean()
rolling_std = pd.Series(y).rolling(window=window).std()

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(t, y, label='Original Series', alpha=0.7, linewidth=1.5)
ax.plot(t, rolling_mean, label='Rolling Mean', color='red', linewidth=2)
ax.plot(t, rolling_std, label='Rolling Std Dev', color='green', linewidth=2)
ax.set_xlabel('Time Step')
ax.set_ylabel('Value')
ax.set_title('Stationarity Check: Original Series (Non-Stationary - Trend Present)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Non-stationary: Mean and variance change over time')


## 7. Differencing to Remove Trend

In [ ]:
# First-order differencing
y_diff = np.diff(y)

# Check stationarity of differenced series
rolling_mean_diff = pd.Series(y_diff).rolling(window=window).mean()
rolling_std_diff = pd.Series(y_diff).rolling(window=window).std()

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Original
axes[0].plot(t, y, alpha=0.7, linewidth=1.5, label='Original')
axes[0].plot(t, rolling_mean, color='red', linewidth=2, label='Rolling Mean')
axes[0].set_title('Original Series (Non-Stationary)')
axes[0].set_ylabel('Value')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Differenced
axes[1].plot(t[1:], y_diff, alpha=0.7, linewidth=1.5, label='First Difference')
axes[1].plot(t[1:], rolling_mean_diff, color='red', linewidth=2, label='Rolling Mean')
axes[1].set_title('After First-Order Differencing (Stationary)')
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('Value')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 8. Exponential Smoothing - Simple

In [ ]:
# Simple exponential smoothing
def simple_exponential_smoothing(series, alpha=0.3):
    result = np.zeros_like(series)
    result[0] = series[0]
    
    for t in range(1, len(series)):
        result[t] = alpha * series[t] + (1 - alpha) * result[t-1]
    
    return result

alphas = [0.1, 0.3, 0.7]
fig, axes = plt.subplots(1, len(alphas), figsize=(15, 4))

for idx, alpha in enumerate(alphas):
    smoothed = simple_exponential_smoothing(y, alpha=alpha)
    axes[idx].plot(t, y, label='Original', alpha=0.5, linewidth=1)
    axes[idx].plot(t, smoothed, label=f'Smoothed (α={alpha})', linewidth=2, color='red')
    axes[idx].set_title(f'Simple Exponential Smoothing (α={alpha})')
    axes[idx].set_xlabel('Time Step')
    axes[idx].set_ylabel('Value')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 9. Exponential Smoothing - Double (Holt's)

In [ ]:
# Double exponential smoothing (Holt's method)
def double_exponential_smoothing(series, alpha=0.3, beta=0.1):
    result = np.zeros_like(series)
    level = series[0]
    trend = series[1] - series[0]
    result[0] = level
    
    for t in range(1, len(series)):
        prev_level = level
        level = alpha * series[t] + (1 - alpha) * (prev_level + trend)
        trend = beta * (level - prev_level) + (1 - beta) * trend
        result[t] = level + trend
    
    return result

simple_smooth = simple_exponential_smoothing(y, alpha=0.3)
double_smooth = double_exponential_smoothing(y, alpha=0.3, beta=0.1)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(t, y, label='Original', alpha=0.5, linewidth=1)
ax.plot(t, simple_smooth, label='Simple Exponential Smoothing', linewidth=2)
ax.plot(t, double_smooth, label="Double Exponential Smoothing (Holt's)", linewidth=2)
ax.set_xlabel('Time Step')
ax.set_ylabel('Value')
ax.set_title('Comparison: Simple vs Double Exponential Smoothing')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 10. Rolling Statistics

In [ ]:
# Compute and plot rolling statistics
rolling_window = 12
rolling_mean = pd.Series(y).rolling(window=rolling_window).mean()
rolling_std = pd.Series(y).rolling(window=rolling_window).std()
rolling_min = pd.Series(y).rolling(window=rolling_window).min()
rolling_max = pd.Series(y).rolling(window=rolling_window).max()

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(t, y, label='Original', alpha=0.5, linewidth=1)
ax.plot(t, rolling_mean, label='Rolling Mean', linewidth=2, color='red')
ax.fill_between(t, rolling_min, rolling_max, alpha=0.2, color='blue', label='Rolling Min-Max')
ax.set_xlabel('Time Step')
ax.set_ylabel('Value')
ax.set_title(f'Rolling Statistics (Window={rolling_window})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 11. Train/Test Split for Time Series

In [ ]:
# Time series train/test split (no shuffling)
train_size = int(0.7 * len(y))
y_train = y[:train_size]
y_test = y[train_size:]
t_train = t[:train_size]
t_test = t[train_size:]

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(t_train, y_train, label='Training Set', linewidth=1.5, color='blue')
ax.plot(t_test, y_test, label='Test Set', linewidth=1.5, color='red')
ax.axvline(x=train_size, color='black', linestyle='--', linewidth=2, label='Train/Test Split')
ax.set_xlabel('Time Step')
ax.set_ylabel('Value')
ax.set_title('Time Series Train/Test Split (No Shuffling)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Train size: {len(y_train)}, Test size: {len(y_test)}')


## Interview Takeaways

1. **Decomposition**: Separate trend, seasonality, and residuals to understand patterns
2. **Stationarity Critical**: Non-stationary series need differencing before modeling
3. **ACF vs PACF**: ACF identifies MA order, PACF identifies AR order
4. **Exponential Smoothing**: Simple for no trend, double for trend, triple for seasonality
5. **Rolling Statistics**: Monitor mean/std changes to assess stationarity
6. **No Shuffling**: Always use time-ordered splits for time series
7. **Walk-Forward Validation**: Better than random CV for time series
8. **Lag Analysis**: ACF/PACF plots guide ARIMA parameter selection